In [ ]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.ml.classification import (LogisticRegression,RandomForestClassifier)
from xgboost.spark import SparkXGBClassifier
from pyspark.ml.evaluation import (MulticlassClassificationEvaluator)
from pyspark.ml.tuning import (ParamGridBuilder,CrossValidator)
from pyspark.sql import functions as F
import time
import pandas as pd
from pyspark.ml.classification import GBTClassifier,OneVsRest
from xgboost.spark import SparkXGBClassifier
from pyspark.mllib.evaluation import MulticlassMetrics
import zipfile
import os
import shutil
from google.colab import files
import numpy as np

In [ ]:
# Create the SparkSession.
spark = (SparkSession.builder.appName("NYC_Taxi_Task3").getOrCreate())

In [ ]:
# paths of ZIP files
TRAIN_ZIP = "/content/train_prepared.zip"
TEST_ZIP = "/content/test_prepared.zip"
with zipfile.ZipFile(TRAIN_ZIP, "r") as zip_ref:
    zip_ref.extractall("/content/train_prepared")
with zipfile.ZipFile(TEST_ZIP, "r") as zip_ref:
    zip_ref.extractall("/content/test_prepared")
print("Training and testing ZIP files extracted successfully.")

Training and testing ZIP files extracted successfully.


In [ ]:
# load the prepared training and testing datasets.
train_prepared = spark.read.parquet("/content/train_prepared")
test_prepared = spark.read.parquet("/content/test_prepared")

In [ ]:
# load the preprocessed data
print("Training dataset")
train_prepared.show(5, truncate=False)
print("Testing dataset")
test_prepared.show(5, truncate=False)
print("Training rows:",f"{train_prepared.count():,}")
print("Testing rows:",f"{test_prepared.count():,}")
train_prepared.printSchema()

Training dataset
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                                                                                                                                       |label|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|(550,[0,2,4,5,8,14,24,47,331],[1.2796361685365383,0.5073004121979604,2.228664870043009,0.2908365368834783,2.3545811223893347,3.3415630662567515,3.3415630662567515,7.153895606048931,11.096568035205054])                      |2.0  |
|(550,[0,2,4,5,8,14,24,53,295],[1.2796361685365383,0.50

In [ ]:
# Evaluation functions
accuracy_evaluator = MulticlassClassificationEvaluator(labelCol="label",predictionCol="prediction",metricName="accuracy")
precision_evaluator = MulticlassClassificationEvaluator(labelCol="label",predictionCol="prediction",metricName="weightedPrecision")
recall_evaluator = MulticlassClassificationEvaluator(labelCol="label",predictionCol="prediction",metricName="weightedRecall")
f1_evaluator = MulticlassClassificationEvaluator(labelCol="label",predictionCol="prediction",metricName="f1")

In [ ]:
def evaluate_model(model_name,predictions,training_time,best_parameters):
    return {
        "Model": model_name,
        "Accuracy": accuracy_evaluator.evaluate(predictions),
        "Precision": precision_evaluator.evaluate(predictions),
        "Recall": recall_evaluator.evaluate(predictions),
        "F1 Score": f1_evaluator.evaluate(predictions),
        "Training Time(sec)": round(training_time,2),
        "Best Parameters": str(best_parameters)
    }

In [ ]:
# Logistic Regression
lr = LogisticRegression(featuresCol="features",labelCol="label",maxIter=50)
lr_grid = (ParamGridBuilder().addGrid(lr.regParam,[0.01,0.1]).addGrid(lr.elasticNetParam,[0.0,0.5]).build())
lr_cv = CrossValidator(estimator=lr, estimatorParamMaps=lr_grid, evaluator=f1_evaluator, numFolds=3,parallelism=2)
start=time.time()
lr_model = lr_cv.fit(train_prepared)
lr_time=time.time()-start
lr_predictions = lr_model.transform(test_prepared)
lr_result=evaluate_model("Logistic Regression",lr_predictions,lr_time,lr_model.bestModel.extractParamMap())
lr_result

{'Model': 'Logistic Regression',
 'Accuracy': 0.840085521049715,
 'Precision': 0.786954436192835,
 'Recall': 0.840085521049715,
 'F1 Score': 0.7738119133796939,
 'Training Time(sec)': 937.73,
 'Best Parameters': "{Param(parent='LogisticRegression_006d5740f503', name='aggregationDepth', doc='suggested depth for treeAggregate (>= 2).'): 2, Param(parent='LogisticRegression_006d5740f503', name='elasticNetParam', doc='the ElasticNet mixing parameter, in range [0, 1]. For alpha = 0, the penalty is an L2 penalty. For alpha = 1, it is an L1 penalty.'): 0.0, Param(parent='LogisticRegression_006d5740f503', name='family', doc='The name of family which is a description of the label distribution to be used in the model. Supported options: auto, binomial, multinomial'): 'auto', Param(parent='LogisticRegression_006d5740f503', name='featuresCol', doc='features column name.'): 'features', Param(parent='LogisticRegression_006d5740f503', name='fitIntercept', doc='whether to fit an intercept term.'): True

In [ ]:
# Random Forest Classifier
rf = RandomForestClassifier(featuresCol="features",labelCol="label",seed=42)
rf_grid=(ParamGridBuilder().addGrid(rf.maxDepth,[2,5]).build())
rf_cv=CrossValidator(estimator=rf,estimatorParamMaps=rf_grid,evaluator=f1_evaluator,numFolds=2,parallelism=2,seed =42)
start=time.time()
rf_model=rf_cv.fit(train_prepared)
rf_time=time.time()-start
rf_predictions=rf_model.transform(test_prepared)
rf_result=evaluate_model("Random Forest",rf_predictions,rf_time,rf_model.bestModel.extractParamMap())
rf_result

{'Model': 'Random Forest',
 'Accuracy': 0.839018533341543,
 'Precision': 0.7068152393203224,
 'Recall': 0.839018533341543,
 'F1 Score': 0.7664296645789858,
 'Training Time(sec)': 416.86,
 'Best Parameters': '{Param(parent=\'RandomForestClassifier_96f1f59223f4\', name=\'bootstrap\', doc=\'Whether bootstrap samples are used when building trees.\'): True, Param(parent=\'RandomForestClassifier_96f1f59223f4\', name=\'cacheNodeIds\', doc=\'If false, the algorithm will pass trees to executors to match instances with nodes. If true, the algorithm will cache node IDs for each instance. Caching can speed up training of deeper trees. Users can set how often should the cache be checkpointed or disable it by setting checkpointInterval.\'): False, Param(parent=\'RandomForestClassifier_96f1f59223f4\', name=\'checkpointInterval\', doc=\'set checkpoint interval (>= 1) or disable checkpoint (-1). E.g. 10 means that the cache will get checkpointed every 10 iterations. Note: this setting will be ignored i

In [ ]:
f1_evaluator = MulticlassClassificationEvaluator(labelCol="label",predictionCol="prediction",metricName="f1")

In [ ]:
# Gradient Boosting using One-vs-Rest
gbt = GBTClassifier(featuresCol="features",labelCol="label",maxIter=2,maxDepth=2,seed=42)
ovr_gbt = OneVsRest(classifier=gbt,featuresCol="features",labelCol="label")
gbt_grid = (ParamGridBuilder().addGrid(ovr_gbt.classifier,[GBTClassifier(featuresCol="features",labelCol="label",maxIter=2,maxDepth=2,seed=42)]).build())
gbt_cv = CrossValidator(estimator=ovr_gbt,estimatorParamMaps=gbt_grid,evaluator=f1_evaluator,numFolds=2,parallelism=1,seed=42)
start = time.time()
gbt_cv_model = gbt_cv.fit(train_prepared)
gbt_time = time.time() - start
gbt_predictions = gbt_cv_model.transform(test_prepared)
gbt_result = evaluate_model("Gradient Boosting",gbt_predictions,gbt_time,gbt_cv_model.bestModel.extractParamMap())
print(gbt_result)

{'Model': 'Gradient Boosting', 'Accuracy': 0.8390242044986859, 'Precision': 0.7068190644350728, 'Recall': 0.8390242044986859, 'F1 Score': 0.7664348065895243, 'Training Time(sec)': 1524.96, 'Best Parameters': "{Param(parent='OneVsRestModel_b0e6394fad8b', name='predictionCol', doc='prediction column name.'): 'prediction', Param(parent='OneVsRestModel_b0e6394fad8b', name='featuresCol', doc='features column name.'): 'features', Param(parent='OneVsRestModel_b0e6394fad8b', name='labelCol', doc='label column name.'): 'label', Param(parent='OneVsRestModel_b0e6394fad8b', name='rawPredictionCol', doc='raw prediction (a.k.a. confidence) column name.'): 'rawPrediction', Param(parent='OneVsRestModel_b0e6394fad8b', name='classifier', doc='base binary classifier'): GBTClassifier_3bdca226777d}"}


In [ ]:
# count the distinct numeric labels actually present in the training data.
num_classes = train_prepared.select("label").distinct().count()
print("Number of classes:", num_classes)

Number of classes: 5


In [ ]:
# XGBoost Classifier
xgb = SparkXGBClassifier(features_col="features",label_col="label",num_workers=1,seed=42)
xgb_grid=(ParamGridBuilder().addGrid(xgb.max_depth,[3,5]).addGrid(xgb.n_estimators,[3]).build())
xgb_cv=CrossValidator(estimator=xgb,estimatorParamMaps=xgb_grid,evaluator=f1_evaluator,numFolds=2,parallelism=1)
start=time.time()
xgb_model=xgb_cv.fit(train_prepared)
xgb_time=time.time()-start
xgb_predictions=xgb_model.transform(test_prepared)
xgb_result=evaluate_model("XGBoost",xgb_predictions,xgb_time,xgb_model.bestModel.extractParamMap())
xgb_result

INFO:XGBoost-PySpark:Running xgboost-3.4.1 on 1 workers with
	booster params: {'device': 'cpu', 'max_depth': 3, 'objective': 'multi:softprob', 'seed': 42, 'num_class': 5, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 3}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!
INFO:XGBoost-PySpark:Running xgboost-3.4.1 on 1 workers with
	booster params: {'device': 'cpu', 'max_depth': 5, 'objective': 'multi:softprob', 'seed': 42, 'num_class': 5, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 3}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!
INFO:XGBoost-PySpark:Running xgboost-3.4.1 on 1 workers with
	booster params: {'device': 'cpu', 'max_depth': 3, 'objective': 'multi:softprob', 'seed': 42, 'num_class': 5, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 3}
	dmatrix_kwargs: {'nthread': 1, 'miss

{'Model': 'XGBoost',
 'Accuracy': 0.8394633140946078,
 'Precision': 0.8277168543107065,
 'Recall': 0.8394633140946077,
 'F1 Score': 0.7677463490713908,
 'Training Time(sec)': 555.52,
 'Best Parameters': "{Param(parent='SparkXGBClassifier_c374156babe0', name='enable_sparse_data_optim', doc='This stores the boolean config of enabling sparse data optimization, if enabled, Xgboost DMatrix object will be constructed from sparse matrix instead of dense matrix. This config is disabled by default. If most of examples in your training dataset contains sparse features, we suggest to enable this config.'): False, Param(parent='SparkXGBClassifier_c374156babe0', name='featuresCol', doc='features column name.'): 'features', Param(parent='SparkXGBClassifier_c374156babe0', name='features_cols', doc='feature column names.'): [], Param(parent='SparkXGBClassifier_c374156babe0', name='labelCol', doc='label column name.'): 'label', Param(parent='SparkXGBClassifier_c374156babe0', name='predictionCol', doc='

In [ ]:
# Model comparison
model_results=[lr_result,rf_result,gbt_result,xgb_result]
results_df=spark.createDataFrame(model_results)
results_df.show(truncate=False)

+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
#  predictions
predictions_by_model = {
    "Logistic Regression": lr_predictions,
    "Random Forest": rf_predictions,
    "Gradient Boosting": gbt_predictions,
    "XGBoost": xgb_predictions}
all_class_results = []
for model_name, predictions in predictions_by_model.items():
    print(f"Evaluating {model_name}")
    pairs = (predictions.select(
            F.col("label").cast("double").alias("actual"),
            F.col("prediction").cast("double").alias("predicted"))
        .dropna()
        .groupBy("actual", "predicted")
        .count().collect())
    class_labels = sorted({
        float(row["actual"]) for row in pairs} | {float(row["predicted"]) for row in pairs})
    class_to_idx = {label: idx for idx, label in enumerate(class_labels)}
    confusion = np.zeros((len(class_labels), len(class_labels)),dtype=np.int64)
    for row in pairs:
        actual_idx = class_to_idx[float(row["actual"])]
        predicted_idx = class_to_idx[float(row["predicted"])]
        confusion[actual_idx, predicted_idx] += row["count"]
    # Rows = actual classes; columns = predicted classes
    tp = np.diag(confusion).astype(float)
    actual_totals = confusion.sum(axis=1).astype(float)
    predicted_totals = confusion.sum(axis=0).astype(float)
    total = confusion.sum()
    for i, class_label in enumerate(class_labels):
        fp = predicted_totals[i] - tp[i]
        fn = actual_totals[i] - tp[i]
        tn = total - tp[i] - fp - fn
        precision = tp[i] / predicted_totals[i] if predicted_totals[i] else 0.0
        recall = tp[i] / actual_totals[i] if actual_totals[i] else 0.0
        f1 = (2 * precision * recall / (precision + recall)
            if precision + recall else 0.0)
        fpr = fp / (fp + tn) if (fp + tn) else 0.0
        all_class_results.append({
            "Model": model_name,
            "Class": int(class_label),
            "Precision": precision,
            "Recall": recall,
            "F1 Score": f1,
            "False Positive Rate": fpr})
class_metrics_df = pd.DataFrame(all_class_results)
display(class_metrics_df.sort_values(["Model", "Class"]).round(4).reset_index(drop=True))

Evaluating Logistic Regression
Evaluating Random Forest
Evaluating Gradient Boosting
Evaluating XGBoost


,Model,Class,Precision,Recall,F1 Score,False Positive Rate
0,Gradient Boosting,0,0.8213,1.0000,0.9019,0.6184
1,Gradient Boosting,1,0.0000,0.0000,0.0000,0.0000
2,Gradient Boosting,2,1.0000,1.0000,1.0000,0.0000
3,Gradient Boosting,3,0.0000,0.0000,0.0000,0.0000
4,Gradient Boosting,4,0.0000,0.0000,0.0000,0.0000
5,Logistic Regression,0,0.8251,0.9967,0.9028,0.6005
6,Logistic Regression,1,0.4929,0.0246,0.0469,0.0039
7,Logistic Regression,2,1.0000,1.0000,1.0000,0.0000
8,Logistic Regression,3,0.3824,0.0005,0.0011,0.0000
9,Logistic Regression,4,0.5109,0.0271,0.0514,0.0002


In [ ]:
model_results_df = pd.DataFrame(model_results)
model_results_df.to_csv("/content/task3_model_results.csv",index=False)
display(model_results_df)

,Model,Accuracy,Precision,Recall,F1 Score,Training Time(sec),Best Parameters
0,Logistic Regression,0.840086,0.786954,0.840086,0.773812,937.73,{Param(parent='LogisticRegression_006d5740f503...
1,Random Forest,0.839019,0.706815,0.839019,0.766430,416.86,{Param(parent='RandomForestClassifier_96f1f592...
2,Gradient Boosting,0.839024,0.706819,0.839024,0.766435,1524.96,"{Param(parent='OneVsRestModel_b0e6394fad8b', n..."
3,XGBoost,0.839463,0.827717,0.839463,0.767746,555.52,{Param(parent='SparkXGBClassifier_c374156babe0...


In [ ]:
# four-model comparison table
model_results = [lr_result,rf_result,gbt_result,xgb_result]
required_columns = ["Model","Best Parameters","Training Time(sec)","Accuracy","Precision","Recall","F1 Score"]
model_results_df = pd.DataFrame(model_results)[required_columns]
# round metrics and training time for clear presentation
metric_columns = ["Training Time(sec)","Accuracy","Precision","Recall","F1 Score"]
model_results_df[metric_columns] = model_results_df[metric_columns].round(4)
display(model_results_df)

,Model,Best Parameters,Training Time(sec),Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,{Param(parent='LogisticRegression_006d5740f503...,937.73,0.8401,0.7870,0.8401,0.7738
1,Random Forest,{Param(parent='RandomForestClassifier_96f1f592...,416.86,0.8390,0.7068,0.8390,0.7664
2,Gradient Boosting,"{Param(parent='OneVsRestModel_b0e6394fad8b', n...",1524.96,0.8390,0.7068,0.8390,0.7664
3,XGBoost,{Param(parent='SparkXGBClassifier_c374156babe0...,555.52,0.8395,0.8277,0.8395,0.7677


In [ ]:
# Save the fitted cross-validation models
MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)
fitted_models = {
    "lr": lr_model,
    "random_forest": rf_model,
    "gradient_boosting": gbt_cv_model,
    "xgboost": xgb_model}
for name, model in fitted_models.items():
    output_path = os.path.join(MODEL_DIR, name)
    model.write().overwrite().save(output_path)
    print(f"Saved {name}: {output_path}")

Saved lr: /content/models/lr
Saved random_forest: /content/models/random_forest
Saved gradient_boosting: /content/models/gradient_boosting
Saved xgboost: /content/models/xgboost
